# LangChain: Agents

大模型本身是无状态的，即参数不会改变，很多人用llm是作为知识库使用的，llm的确存储了很多知识

但是同时llm具有很强的推理能力，可以为llm提供一些信息让llm来回答问题

## Outline:

* Using built in LangChain tools: DuckDuckGo search and Wikipedia
* Defining your own tools

In [1]:
# 初始化
from langchain.chat_models import ChatOpenAI
import os
API_SECRET_KEY = "sk-scia8t0NThqBoFbDdzweXuPRmeQrMuQ9XkJaYl29VHyMACFD"
BASE_URL = "https://api.chatanywhere.tech"
os.environ["OPENAI_API_KEY"] = API_SECRET_KEY
os.environ["OPENAI_API_BASE"] = BASE_URL
llm_model = "gpt-3.5-turbo"
import warnings
warnings.filterwarnings("ignore")

## 内置LangChain工具创建Agent例子

In [ ]:
#!pip install -U wikipedia
from langchain.agents.agent_toolkits import create_python_agent
from langchain.agents import load_tools, initialize_agent
from langchain.agents import AgentType
from langchain.tools.python.tool import PythonREPLTool
from langchain.python import PythonREPL
from langchain.chat_models import ChatOpenAI


llm = ChatOpenAI(temperature=0, model=llm_model) # 我们要用来精确的推理而不是写作，所以热度为0
# 加载两个工具，llm-math是一个chain，结合了llm和计算器 来解决数学问题
# wikipedia是一个连接到维基百科的程序，输入查询内容，返回结果
tools = load_tools(["llm-math","wikipedia"], llm=llm) 
# 实际工作时，就和routeChain一样，先输入问题让llm得出相关调用的工具，自动选择下一个chain，如果是数学问题，将调用计算器工具得到答案
# 如果是wikipedia问题，就去查询维基结果
# 然后将两个工具的结果输入下一个chain的模板中的observation
# 下一个chain再去问llm得到最后的thought部分结果

In [3]:
# 初始化一个agent
agent= initialize_agent(
    tools, #工具列表
    llm,   # 模型
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION, # 代理类型 CHAT：为chat模型优化 REACT：一种组织prompt技术，最大化推理能力 
    handle_parsing_errors=True, #llm输出无法解析时 将错误内容重新给llm让它修正
    verbose = True)

### 数学问题例子

In [ ]:
# 对agent提出问题
agent("What is the 25% of 300?")

# 观察一下结果：Answer实际上是调用计算器的结果，thought才是llm的输出



> Entering new AgentExecutor chain...
Thought: We can use a calculator to find 25% of 300. 

Action:
```
{
  "action": "Calculator",
  "action_input": "25% of 300"
}
```


Observation: Answer: 75.0
Thought:Final Answer: 75.0

> Finished chain.


{'input': 'What is the 25% of 300?', 'output': '75.0'}

### Wikipedia 例子

In [6]:
question = "Tom M. Mitchell is an American computer scientist \
and the Founders University Professor at Carnegie Mellon University (CMU)\
what book did he write?"
result = agent(question) 



> Entering new AgentExecutor chain...
Thought: I should use Wikipedia to find out which book Tom M. Mitchell wrote.
Action:
```
{
  "action": "Wikipedia",
  "action_input": "Tom M. Mitchell"
}
```
Observation: Page: Tom M. Mitchell
Summary: Tom Michael Mitchell (born August 9, 1951) is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU). He is a founder and former chair of the Machine Learning Department at CMU. Mitchell is known for his contributions to the advancement of machine learning, artificial intelligence, and cognitive neuroscience and is the author of the textbook Machine Learning. He is a member of the United States National Academy of Engineering since 2010. He is also a Fellow of the American Academy of Arts and Sciences, the American Association for the Advancement of Science and a Fellow and past president of the Association for the Advancement of Artificial Intelligence. In October 2018, Mitchell was appointed as t

## 使用Agent来做一个Copilot应用

任务：输入一组客户名单，给名单排序
要求：先按姓氏再按名字排序

In [9]:
# 定义agent，给出工具和llm，PythonREPLTool是一种代码互动方式，类似jupyter，调用者输入代码，得到运行结果，结果传给agent，让它决定接下来做什么
agent = create_python_agent(
    llm,
    tool=PythonREPLTool(),
    verbose=True
)
# 名单
customer_list = [["Harrison", "Chase"], 
                 ["Lang", "Chain"],
                 ["Dolly", "Too"],
                 ["Elle", "Elem"], 
                 ["Geoff","Fusion"], 
                 ["Trance","Former"],
                 ["Jen","Ayai"]
                ]
#结合排序要求的提示词
agent.run(f"""Sort these customers by \
last name and then first name \
and print the output: {customer_list}""") 
# run的结果会反馈会agent



> Entering new AgentExecutor chain...
We can use the `sorted()` function in Python to sort the list of customers based on last name first and then first name.
Action: Python REPL
Action Input: customers = [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]
sorted_customers = sorted(customers, key=lambda x: (x[1], x[0]))
print(sorted_customers)
Observation: [['Jen', 'Ayai'], ['Lang', 'Chain'], ['Harrison', 'Chase'], ['Elle', 'Elem'], ['Trance', 'Former'], ['Geoff', 'Fusion'], ['Dolly', 'Too']]

Thought:The customers are now sorted by last name and then first name.
Final Answer: [['Jen', 'Ayai'], ['Lang', 'Chain'], ['Harrison', 'Chase'], ['Elle', 'Elem'], ['Trance', 'Former'], ['Geoff', 'Fusion'], ['Dolly', 'Too']]

> Finished chain.


"[['Jen', 'Ayai'], ['Lang', 'Chain'], ['Harrison', 'Chase'], ['Elle', 'Elem'], ['Trance', 'Former'], ['Geoff', 'Fusion'], ['Dolly', 'Too']]"

#### 查看Agent底层工作机制

全局debug打开为true再跑一遍

In [ ]:
import langchain
langchain.debug=True
agent.run(f"""Sort these customers by \
last name and then first name \
and print the output: {customer_list}""") 
langchain.debug=False

# 观察结果
# 1. 第一个chain接受输入文本，然后调用第二个LLMchain
# 2. 第二个LLMChain只将prompt结合进去，使用一个json格式，并且在stop字段告诉llm什么时候工作停止，然后调用第三个Chain来访问llm
# 3. 第三个Chain完善prompt，包括可以访问哪些工具，用什么格式输出，
#           然后调用LLM，得到llm输出后退出并回到第一个chain
# 4. 第一个chain 根据第三个llm输出的 action选择工具并调用，根据inputs作为输入代码并执行
# 5. 第四个是工具，执行完毕代码后获得输出退回第一个chain
# 6. 第一个chain的agent_scratchpad内的observation保存了执行的结果，然后继续进入第五个LLMChain，将所有推理过程发送llm，这时候llm意识到工作已完成
# 7. llm输出格式化的结果



[chain/start] [1:chain:AgentExecutor] Entering Chain run with input:
{
  "input": "Sort these customers by last name and then first name and print the output: [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]"
}
[chain/start] [1:chain:AgentExecutor > 2:chain:LLMChain] Entering Chain run with input:
{
  "input": "Sort these customers by last name and then first name and print the output: [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]",
  "agent_scratchpad": "",
  "stop": [
    "\nObservation:",
    "\n\tObservation:"
  ]
}
[llm/start] [1:chain:AgentExecutor > 2:chain:LLMChain > 3:llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: You are an agent designed to write and execute python code to answer questions.\nYou have access to a python REPL, which you can use to execute python code.\nI

## Define your own tool

In [ ]:
#!pip install DateTime

In [11]:
from langchain.agents import tool
from datetime import date

In [ ]:
@tool
def time(text: str) -> str:
    # 告知什么时候可以使用这个tool 并说明输入格式，输出格式
    """Returns todays date, use this for any \
    questions related to knowing todays date. \
    The input should always be an empty string, \
    and this function will always return todays \
    date - any date mathmatics should occur \
    outside this function."""
    return str(date.today())

In [13]:
agent= initialize_agent(
    tools + [time], 
    llm, 
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose = True)

**Note**: 

The agent will sometimes come to the wrong conclusion (agents are a work in progress!). 

If it does, please try running it again.

In [14]:
try:
    result = agent("whats the date today?") 
except: 
    print("exception on external access")



> Entering new AgentExecutor chain...
Thought: I can use the `time` tool to find out today's date.
Action:
```
{
  "action": "time",
  "action_input": ""
}
```

Observation: 2024-12-15
Thought:
Observation: Invalid or incomplete response
Thought:I will correct the previous action to get today's date properly.

Action:
```
{
  "action": "time",
  "action_input": ""
}
```

Final Answer: 2024-12-15

> Finished chain.
